In [21]:
import pandas as pd
import matplotlib.pyplot as plt
import shutil
import os

from shared import calculate_summer_avg

In [22]:
target_regions=[
    "Northeast Region",
    "East North Central Region",
    "Central Region",
    "Southeast Region",
    "West North Central Region",
    "South Region",
    "Southwest Region",
    "Northwest Region",
    "West Region",
    "(Contiguous 48 States)",
]

target_states=[
    "Arizona", 
    "Nevada", 
    "New Mexico", 
    "Utah",
    "California"
]
drought_df=pd.read_excel("US-Annual-Precipitation-By-State.xlsx",sheet_name="Palmer_Drought_Severity_Index")
drought_df["Summer_Avg"]=drought_df[['Jun','Jul','Aug']].mean(axis=1)

precipitation_df=pd.read_excel("US-Annual-Precipitation-By-State.xlsx",sheet_name="Precipitation_Data")
precipitation_df["Summer_Avg"]=precipitation_df.apply(calculate_summer_avg,axis=1)

In [23]:
def create_plot(orig_df,target_column:str,target_folder,min_y,max_y):
    out_folder=f"output/{target_folder}/{target_column.lower()}"
    try:
        shutil.rmtree(out_folder)
    except FileNotFoundError:
        pass
    finally:
        os.mkdir(out_folder)
    
    target_folder_written:str=target_folder
    target_folder_written=target_folder_written.replace("_"," ")
    target_folder_written=target_folder_written.title()

    if target_column=="State":
        orig_df=orig_df[~pd.isna(orig_df["Region"])]
    else:
        orig_df=orig_df[pd.isna(orig_df["Region"])]

    unique_values=list(orig_df["State"].unique())
    for value in unique_values:
        orig_df_filtered=orig_df[orig_df["State"]==value]
        orig_df_filtered = orig_df_filtered.groupby(['Decade'], as_index=False)['Summer_Avg'].mean()

        fig, ax = plt.subplots()
        x=orig_df_filtered["Decade"]
        y=orig_df_filtered["Summer_Avg"]
        ax.plot(x,y)
        for xi,yi in zip(x,y):
            ax.annotate(
                str(round(yi,1)), 
                xy=(xi, yi),                # Point to annotate
                xytext=(0, 8),              # Offset text by 8 points vertically
                textcoords="offset points", # Use point-based offset
                ha='center',                # Horizontally center text over the point
                va='bottom'                 # Position text baseline above the point
            )
        
        plt.title(f"{value} {target_folder_written}\nBy Decade")

        plt.ylim(min_y,max_y)
        plt.savefig(f"{out_folder}/{value}.png")
        plt.close()


In [24]:
create_plot(drought_df,"State","palmer_drought_index_charts",-4,4)
create_plot(drought_df,"Region","palmer_drought_index_charts",-4,4)
create_plot(precipitation_df,"State","precipitation_charts",0,9)
create_plot(precipitation_df,"Region","precipitation_charts",0,9)